In [ ]:
import os
from pathlib import Path
from types import SimpleNamespace

import numpy as np

try:
    from scipy import ndimage, signal, optimize, interpolate, stats
except Exception:
    ndimage = signal = optimize = interpolate = stats = None

def n_elements(x):
    if x is None:
        return 0
    try:
        return np.size(x)
    except Exception:
        return 1

def read_parameter_file(parfile):
    params = {}
    path = Path(parfile)
    if not path.exists():
        raise FileNotFoundError(parfile)

    for raw in path.read_text().splitlines():
        line = raw.split(';', 1)[0].strip()
        if not line or '=' not in line:
            continue
        key, value = line.split('=', 1)
        key = key.strip().lower()
        value = value.strip()
        value = value.replace('[', 'np.array([').replace(']', '])')
        value = value.replace('^', '**')
        try:
            params[key] = eval(value, {'np': np, 'array': np.array})
        except Exception:
            params[key] = value.strip("'\"")
    return params

def write_idl_array_line(f, name, arr, comment=''):
    arr = np.asarray(arr).ravel()
    values = ','.join(f'{v:10.4E}' if abs(v) >= 1e4 or (abs(v) < 1e-3 and v != 0) else f'{v:10.4f}' for v in arr)
    f.write(f'{name.upper()}=[{values}]')
    if comment:
        f.write(f' ; {comment}')
    f.write('\n')

def robust_sigma(values):
    values = np.asarray(values, dtype=float)
    med = np.nanmedian(values)
    return 1.4826 * np.nanmedian(np.abs(values - med))

def linear_interp(x, y, x_new):
    return np.interp(x_new, np.asarray(x, dtype=float), np.asarray(y, dtype=float))

def congrid(array, new_shape):
    array = np.asarray(array, dtype=float)
    if ndimage is None:
        raise ImportError('scipy is required for congrid')
    zoom = [n / o for n, o in zip(new_shape, array.shape)]
    return ndimage.zoom(array, zoom, order=1)

def idl_hist2d(x, y, xbin, ybin, xmin, xmax, ymin, ymax):
    x_edges = np.arange(xmin, xmax + xbin, xbin)
    y_edges = np.arange(ymin, ymax + ybin, ybin)
    hist, _, _ = np.histogram2d(x, y, bins=[x_edges, y_edges])
    return hist

def load_model_file(path):
    with open(path, 'r') as f:
        age_line = f.readline().strip()
        feh_line = f.readline().strip()
        shape_line = f.readline().strip()
        shape = tuple(int(v) for v in shape_line.replace(',', ' ').split()[:2])
        data = np.loadtxt(f)
    return age_line, feh_line, data.reshape(shape)

def save_model_file(path, age_label, feh_label, model):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    model = np.asarray(model, dtype=float)
    with open(path, 'w') as f:
        f.write(age_label + '\n')
        f.write(feh_label + '\n')
        f.write(f'{model.shape[0]} {model.shape[1]}\n')
        np.savetxt(f, model)


In [ ]:
def mcanal_m31(sfhout, nmc=100):
    sfrin, sfrerr, age, agesz = mcanal(sfhout, nmc=nmc)
    outfile = Path(sfhout).stem
    sfrmc = np.zeros((len(sfrin), nmc))
    for ss in range(1, nmc + 1):
        mc_file = Path('montec') / f'{outfile}{ss}.dat'
        if mc_file.exists():
            sfrmc[:, ss - 1] = np.asarray(readsfh(mc_file)['sfr'], dtype=float)
    sfrerr2 = np.std(sfrmc, axis=1, ddof=1)
    with open(sfhout, 'a') as f:
        write_idl_array_line(f, 'SFRERR2', sfrerr2, 'M_sun yr^-1')
    return sfrin, sfrerr, sfrerr2, age, agesz
